# Sharded image training

The corpus is generated here. The pipeline then splits, packs, trains, and evaluates as four separate steps.

1. Generate 20,000 labeled 256×256 overhead JPEGs (truck, boat, plane, building) on the shared volume.
2. Preview a 4×4 grid. Each class varies in size, heading, and position.
3. `split-dataset` writes a stratified 80/20 split.
4. `pack-shards` writes mixed WebDataset shards under `/data/shards/train` and `/data/shards/test`.
5. `train` submits a two-GPU TrainJob that fits ResNet18 for one epoch and saves a checkpoint.
6. `evaluate` submits a second TrainJob that scores the held-out shards.
7. The last cell registers pipeline `overhead-shard-training` version `v2` and starts a run from that version. Open **Develop and train → Pipelines**. The pipeline experiment and the MLflow experiment are both named `overhead-shard-training`. Metrics, the confusion matrix, and GPU utilization are on the nested run for each step. The parent run stays empty.

One epoch and 20,000 images are the demo defaults. If this folder already holds a larger corpus, the generate cell stops instead of deleting it. `scaling-to-one-million.md` lists what to change for a larger run.

In [ ]:
from generate_images import CLASSES, count_images, generate

RAW = "/opt/app-root/src/data/raw"
generate(20000, RAW)
print("classes:", ", ".join(CLASSES))
print("image_count:", count_images(RAW))

In [ ]:
from generate_images import CLASSES, sample_grid
from PIL import Image
import matplotlib.pyplot as plt

paths = sample_grid(RAW, per_class=4)
fig, axes = plt.subplots(len(CLASSES), 4, figsize=(8, 8))
for ax, path in zip(axes.ravel(), paths):
    ax.imshow(Image.open(path))
    ax.set_title(path.parent.name)
    ax.axis("off")
for ax in axes.ravel()[len(paths):]:
    ax.axis("off")
fig.tight_layout()
plt.show()

## Prep

`split_dataset.py` reads `/data/raw/<class>/*.jpg` and the class list in `classes.txt`.
It shuffles a stratified 80/20 split and writes that list as JSON. `pack_shards.py`
reads the list and writes mixed WebDataset shards to `/data/shards/train` and
`/data/shards/test`. Each sample is `000123.jpg` plus `000123.cls`. The manifest
records the class list, both shard lists, and the split counts. The image bytes
stay on the shared volume.

## Train

`train` and `evaluate` each call `submit_trainjob.py`, which creates a TrainJob on
runtime `torch-distributed-cuda130-torch211-py312`. Two nodes, one GPU each, run
`train.py`. The train phase fits ResNet18 from scratch for one epoch (DDP, head sized
from the manifest, WebDataset `split_by_node` / `split_by_worker`, `num_workers=2`,
batch 128) and saves `/data/models/resnet18.pt`. The evaluate phase loads that
checkpoint and scores the test shards.

The train step logs one MLflow point per epoch for loss, accuracy, and precision,
recall, and F1. The evaluate step logs the held-out scores and a confusion matrix.
Each training pod samples GPU utilization, and the pipeline task writes that series
under MLflow System metrics. The GPU pods do not call MLflow. The last cell uses
the pipeline defaults, so it trains every image under `/data/raw` for one epoch and
rewrites `/data/shards`.

In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--index-url", "https://pypi.org/simple", "kfp==2.17.0"])
from pipeline import submit

submit()